Carga Inicial de los Dataset Proprocionados

In [62]:
import numpy as np
import pandas as pd
import streamlit as st

df_customers_all = pd.read_csv('../documents/customers_dataset.csv') # Correcto
df_order_items_all = pd.read_csv('../documents/order_items_dataset.csv') # Correcto
df_order_payments_all = pd.read_csv('../documents/order_payments_dataset.csv') # Correcto
df_order_reviews_all = pd.read_csv('../documents/order_reviews_dataset.csv') # Correcto
df_orders_all = pd.read_csv('../documents/orders_dataset.csv') # Correcto
df_product_category_name_translation_all = pd.read_csv('../documents/product_category_name_translation.csv') # Correcto
df_products_all = pd.read_csv('../documents/products_dataset.csv')
df_sellers_all = pd.read_csv('../documents/sellers_dataset.csv') # Correcto

Ejercicio 3.1. Número de pedidos que llegan tarde por ciudad

In [72]:
df_orders_all['order_delivered_customer_date'] = pd.to_datetime(df_orders_all['order_delivered_customer_date'])
df_orders_all['order_estimated_delivery_date'] = pd.to_datetime(df_orders_all['order_estimated_delivery_date'])

days_measurement = (df_orders_all['order_delivered_customer_date'] - df_orders_all['order_estimated_delivery_date']).dt.days

df_late_orders = df_orders_all[
    (df_orders_all['order_delivered_customer_date'] > df_orders_all['order_estimated_delivery_date']) 
    & (df_orders_all['order_status'] == 'delivered') & (days_measurement > 0)
]

df_late_orders_city = pd.merge(df_late_orders, df_customers_all, on='customer_id')

df_late_orders_city.groupby('customer_city').size().sort_values(ascending=False).to_frame().reset_index().rename(columns={'customer_city' : 'Ciudad', 0 : 'Cant. Pedidos'}).head(n=25)

,Ciudad,Cant. Pedidos
0,sao paulo,715
1,rio de janeiro,706
2,salvador,174
3,belo horizonte,137
4,porto alegre,136
5,campinas,119
6,brasilia,118
7,niteroi,96
8,fortaleza,94
9,sao goncalo,83


Ejercicio 3.2. Porcentaje de pedidos retrasados respecto al total de pedidos de la ciudad

In [73]:
df_customers_orders_all = pd.merge(df_orders_all, df_customers_all, on='customer_id')

df_orders_percentage = pd.merge(df_customers_orders_all.groupby('customer_city').size().reset_index(name='total_orders'),
                                df_late_orders_city.groupby('customer_city').size().reset_index(name='total_late_orders'), on='customer_city', how='left').fillna(0)

df_orders_percentage['percentage'] = round((df_orders_percentage['total_late_orders'] / 
                                      df_orders_percentage['total_orders']) * 100, 2)

df_orders_percentage.sort_values(by='percentage', ascending=[False]).reset_index().rename(columns={'customer_city' : 'Ciudad', 'percentage' : 'Porcentaje'}).head(n=25)

,index,Ciudad,total_orders,total_late_orders,Porcentaje
0,1074,corrego fundo,1,1.0,100.0
1,3934,tururu,1,1.0,100.0
2,4013,vargem grande do soturno,1,1.0,100.0
3,3995,valao do barro,1,1.0,100.0
4,1167,desembargador otoni,1,1.0,100.0
5,1326,felicio dos santos,1,1.0,100.0
6,1380,fragosos,1,1.0,100.0
7,3779,tabuleiro do norte,1,1.0,100.0
8,1454,goncalves dias,1,1.0,100.0
9,1208,dom macedo costa,1,1.0,100.0


Ejercicio 3.3 Tiempo medio de retraso en días

In [74]:
df_mean_time_days = df_late_orders_city.copy()
df_mean_time_days['order_delivered_customer_date'] = df_mean_time_days['order_delivered_customer_date'].astype('date64[pyarrow]')
df_mean_time_days['order_estimated_delivery_date'] = df_mean_time_days['order_estimated_delivery_date'].astype('date64[pyarrow]')

df_mean_time_days['late_days'] = (df_mean_time_days['order_delivered_customer_date'] - df_mean_time_days['order_estimated_delivery_date']).dt.days

df_mean_time_days.groupby('customer_city')['late_days'].mean().sort_values(ascending=False).to_frame()

,late_days
customer_city,
montanha,181.0
perdizes,162.0
teutonia,153.0
formosa,152.0
macapa,144.0
...,...
sidrolandia,1.0
teotonio vilela,1.0
tururu,1.0


Determinar causa de insatisfacción de los clientes

In [78]:
df_late_orders_reviews = pd.merge(df_mean_time_days, df_order_reviews_all, on='order_id', how='left')

df_late_orders_reviews[df_late_orders_reviews['review_score'] == 5.0]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,late_days,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
6,a5474c0071dd5d1074e12d417078bbd0,ef15b3240b2083e0487762ee2978d2b8,delivered,2018-07-30 22:41:44,2018-08-01 02:50:15,2018-08-02 10:35:00,2018-08-03,2018-08-02,0c8761c28faa9b9930e4a9bb545905da,6434,barueri,SP,1,ef45857b2a4924df76832d06cca3231d,5.0,Ótimo,Chegou antes do prazo e o produto é exatamente...,2018-08-04 00:00:00,2018-08-06 12:41:29
18,690199d6a2c51ff57c6b392d7680cbfd,19bacb562bd43bd4eaf05b6c0a59dad0,delivered,2018-03-16 11:31:18,2018-03-16 11:49:14,2018-03-19 19:56:23,2018-05-14,2018-04-11,c5f01991eadc43c924dfd891e9026217,87323,alto sao joao,PR,33,aa0aefb73af27de135df1f14af576264,5.0,NaN,NaN,2018-04-13 00:00:00,2018-04-17 17:23:15
25,9d513821c0477231fc7c1bfd684d13d8,00a6e2753fc2652cb87386ffbf5792b2,delivered,2017-02-20 21:31:59,2017-02-22 06:50:09,2017-02-23 07:23:34,2017-03-21,2017-03-17,3abdf4f27efce96f8573a88343a4084e,4944,sao paulo,SP,4,230eae5559b410b3a71bb91b587f9bb2,5.0,NaN,NaN,2017-03-22 00:00:00,2017-03-22 23:22:10
30,4d2d8a4224215f680cc5221011347401,5957d9ae822bcbe88c00fb8286362339,delivered,2018-03-01 12:43:37,2018-03-02 02:10:25,2018-03-02 19:08:11,2018-04-05,2018-04-02,30cf132fa42d1d4434e6b51827b81491,96206,rio grande,RS,3,c93db0ad557b38936732feda0e0143c8,5.0,NaN,NaN,2018-04-05 00:00:00,2018-04-06 23:25:53
43,8ad3f1d0f96992e43566c4c82c9f6c58,948b29e24216a05fea13a18d8db45ea5,delivered,2018-07-17 21:25:29,2018-07-17 21:35:17,2018-07-18 13:08:00,2018-08-14,2018-08-03,6740f8899f3c70b5b08b2e0bad37e567,83252,ilha dos valadares,PR,11,2f8295016240ed6439e0f09c12d5fdd8,5.0,5 estrela bom,Ótimo atendimento,2018-08-05 00:00:00,2018-08-07 13:50:24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6498,30e934394c047a409bb861a9ecbcff43,bfc48761221b04268c51e10b30855efa,delivered,2018-08-09 13:59:34,2018-08-09 14:30:09,2018-08-14 13:09:00,2018-08-17,2018-08-16,c913a62d8196bbf29be088eaf337ce42,2882,sao paulo,SP,1,18fce76a642c2f6249b70f8c47ec99b8,5.0,Muito satisfeita,Muito satisfeita com a minha compra !!!,2018-08-18 00:00:00,2018-08-20 14:09:39
6503,3adb141ba4bd69dd7fe8d3fb733c6b74,c0539d5c87fc7c97a8418adffe4b45f0,delivered,2018-08-14 23:29:21,2018-08-16 03:05:11,2018-08-16 13:28:00,2018-08-28,2018-08-24,7c5750a9ae14793d13ed61b94694963b,21831,rio de janeiro,RJ,4,3e0ec9628cc4d9e73daac3de6c416ea3,5.0,NaN,NaN,2018-08-29 00:00:00,2018-08-29 22:21:36
6507,81fdc868fd0f5c56b9ebea53765e4bbc,d709132e88f6504052afb042f4b7649d,delivered,2018-03-12 11:03:24,2018-03-13 03:50:43,2018-03-16 21:28:20,2018-04-03,2018-04-02,5b8d3d205894a699082c1d6622f69fa9,87033,maringa,PR,1,323874df853b6237e7b4eafcbec51a90,5.0,NaN,NaN,2018-04-04 00:00:00,2018-04-06 16:31:11
6544,25f6cb0e242a4c0bacc239397614422e,c3e28a954146468597f1dfe3ab014eab,delivered,2017-11-20 12:50:49,2017-11-21 04:06:25,2017-11-27 22:04:13,2017-12-22,2017-12-21,e087f21fb7bea52503db8303d3317ae5,55200,pesqueira,PE,1,9c1f4bd9007b7a17865214b2c6231e49,5.0,NaN,NaN,2017-12-22 00:00:00,2017-12-26 00:01:30
